# LD-1B — diagnostic par distance au bord (CPU, mf_0045-0049, checkpoint final LD-1)

In [ ]:

import hashlib, json, os, subprocess, sys
from pathlib import Path
import datetime, faulthandler
faulthandler.enable()
OUT = Path("/kaggle/working")
KIT_CANDIDATES = ['/kaggle/input/babaorum-ld1b-kit', '/kaggle/input/datasets/mabaku/babaorum-ld1b-kit']
LD1_RUN_CANDIDATES = ['/kaggle/input/babaorum-ld1-run/ld1', '/kaggle/input/notebooks/mabaku/babaorum-ld1-run/ld1']
LD1V_CANDIDATES = ['/kaggle/input/babaorum-ld1-v/ld1v/run', '/kaggle/input/notebooks/mabaku/babaorum-ld1-v/ld1v/run']
PAGES_CANDIDATES = ['/kaggle/input/babaorum-e2e3-run3-fp-classes/e2e3/mf', '/kaggle/input/notebooks/mabaku/babaorum-e2e3-run3-fp-classes/e2e3/mf']
TIMEOUTS = {'D': 900, 'DIAG': 10800}

def resolve_one(candidates, marker):
    """Exactement un candidat dont <candidat>/<marqueur> existe. Aucun listage."""
    hits = [Path(c) for c in candidates if (Path(c) / marker).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"montage : {len(hits)} candidat(s) pour {marker}")
    return hits[0]

def resolve_file(candidates):
    """Exactement un candidat qui est un fichier (§12 (15))."""
    hits = [Path(c) for c in candidates if Path(c).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"fichier : {len(hits)} candidat(s)")
    return hits[0]


KIT = resolve_one(KIT_CANDIDATES, "config/ld1b_border.json")
LD1_DIR = resolve_one(LD1_RUN_CANDIDATES, "ld1_timing_manifest.json")
LD1V_DIR = resolve_one(LD1V_CANDIDATES, "ld1_report.json")
PAGES = resolve_one(PAGES_CANDIDATES, "pages/mf_0000.json")
sys.path.insert(0, str(KIT / "src"))
sys.path.insert(0, str(KIT / "scripts"))
import importlib.util as _u
_spec = _u.spec_from_file_location("data2_common", KIT / "scripts" / "data2_common.py")
C = _u.module_from_spec(_spec); _spec.loader.exec_module(C)
CACHE = "/tmp/ld1b_hf"
DIAG_OUT = OUT
STEPS = [
    {"name": "D", "argv": [sys.executable, str(KIT / "scripts" / "train_line_detr.py"), "--phase", "D",
                            "--out", str(OUT / "ld1b_download"), "--cache-dir", CACHE], "timeout_s": TIMEOUTS["D"]},
    {"name": "DIAG", "argv": [sys.executable, str(KIT / "scripts" / "ld1b_border_diag.py"),
                              "--ld1-dir", str(LD1_DIR), "--ld1-report", str(LD1V_DIR / "ld1_report.json"), "--root", str(PAGES), "--out", str(DIAG_OUT),
                              "--cache-dir", CACHE], "timeout_s": TIMEOUTS["DIAG"],
     "env": (lambda: dict(os.environ, HF_HUB_OFFLINE="1"))},
]
EXIT = None
try:
    EXIT = C.execute_pipeline(OUT, STEPS, heartbeat_interval_s=60.0)
finally:
    (OUT / "ld1b_run_exit.json").write_text(json.dumps(EXIT if EXIT is not None else {"status": "FAILED_BEFORE_EXIT"},
                                                      indent=1, default=str))
print("LD1B", EXIT["status"] if EXIT else "none", flush=True)
